# 0/ Import libraries

In [1]:
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pprint
import pickle

# 1/ Load data

## Dataframe

In [2]:
path= "../../data/processed/eeg/df_eeg.parquet"
eeg = pd.read_parquet(path)

In [ ]:
print("initial df shape:", eeg.shape)
eeg.head(5)

# initial df shape: (198023, 2140)

In [4]:
# Non-metrics columns categories:
id_cols = [ 'source_file', 'segment_name', 'is-segment', 'is-bin',  'stimuli_name_clean', 'stimuli_type', 'stimuli_type_order']

temporal_cols = [ 'start_ms', 'end_ms',  'bin_param_duration', 'duration', 'bin_order', 'ad_order',]

dupplicate_cols = [
        'ad_pod_duration_s',
        'device',
        '< RemovedDueToNDA >_clean',
        'age',
        'age_group',
        'gender',
        'is_duplicated_eeg',
        'ad-duration',
        'unaided-brand-recall',
        'aided-brand-recall',
        'cat-recall'
        ]

dummy_cols = [
        'birth_year', 'is_maschio', 'is_femmina',
        'is_smartphone', 'is_television', 'is_laptop', 'is_pod_60_s'
        ]

#### Drop duplicates columns

In [5]:
eeg.drop(columns=dupplicate_cols, inplace=True)

# dummy variables in EEG
eeg.drop(columns= dummy_cols,
        inplace=True)

#### Fixing the main eeg dataframe for issues

In [6]:
# syncing brands columns to merge dataframes
brands_correction_mapping = {
    '< RemovedDueToNDA >': '< RemovedDueToNDA >',
    '< RemovedDueToNDA >': '< RemovedDueToNDA >',
    '< RemovedDueToNDA >': '< RemovedDueToNDA >'
}
eeg['stimuli_name_clean'] = eeg['stimuli_name_clean'].replace(brands_correction_mapping)

# syncing key columns to merge dataframes
eeg['key'] = eeg['source_file'].str.replace("D:\\< RemovedDueToNDA >_AssoTV\\sensor-data\\", "E:\\AssoTV\\sensor-data\\")

# 2/ EEG Data Separation


## Separation of bins and segments

In [7]:
print("initial df shape:", eeg.shape)

# === bins ===
bins = eeg[eeg['is-bin']==1].copy()
print("number of duplicated bins:", bins.duplicated().sum())
bins.drop_duplicates(inplace=True)
print("initial bins shape:", bins.shape)
# drop bins/segment column
bins.drop(
        columns=[ 'is-bin', 'is-segment', 'segment_name',  #drop unnecessary columns
                ],
        inplace=True
        )

print("="*80 )

segments = eeg[eeg['is-segment']==1].copy()
print("number of duplicated segments:", segments.duplicated().sum())
segments.drop_duplicates(inplace=True)
print("initial segments shape:", segments.shape)
# drop bins/segment column
segments.drop(
        columns=['is-bin', 'is-segment', 'segment_name', 'bin_order', # drop unnecessary columns
                'bin_param_duration', 'ad_order', # already exist in ad_break_level_obt
                ], 
        inplace=True
        )

initial df shape: (198023, 2123)
number of duplicated bins: 0
initial bins shape: (193888, 2123)
number of duplicated segments: 0
initial segments shape: (4135, 2123)


# 3/ Segments

In [ ]:
segments.head()

#### Initial inspection

In [9]:
segments.info()
# dtypes: Int64(12), float64(2100), object(3)

# segments.select_dtypes(include=['object']).columns

<class 'pandas.core.frame.DataFrame'>
Index: 4135 entries, 0 to 197954
Columns: 2117 entries, source_file to key
dtypes: Int64(12), float64(2100), object(5)
memory usage: 66.9+ MB


In [10]:
print("Number of missing values in segments dataframe in first 15 columns:")
segments.iloc[:, :15].isna().sum()[segments.isna().sum() > 0]

Number of missing values in segments dataframe in first 15 columns:


mean-psd_Delta_F7_v^2/Hz      3
mean-psd_Delta_AF7_v^2/Hz     8
mean-psd_Delta_Fp1_v^2/Hz     8
mean-psd_Delta_Fpz_v^2/Hz     7
mean-psd_Delta_Fp2_v^2/Hz     7
mean-psd_Delta_AF8_v^2/Hz    10
mean-psd_Delta_F8_v^2/Hz      5
mean-psd_Theta_F7_v^2/Hz      3
dtype: int64

> no missing values found in object columns

In [11]:
print("Number of duplicated rows in segments dataframe:")
print(segments.duplicated().sum())

Number of duplicated rows in segments dataframe:
0


#### Null Values Analysis

In [12]:
print(f"{(segments.isna().sum() > 0).sum()} columns with missing values in segments dataframe.")
print(f"Percentage of rows with missing values in segments: {(segments.isna().any(axis=1).sum() / len(segments) * 100):.2f}%")

segments.dropna(axis=0, inplace=True)
print('-'*80)
print("As null rows are less than 1%, Dropped rows with missing values in segments dataframe.")
print(f"New shape of segments after dropping rows with missing values: {segments.shape}")

2109 columns with missing values in segments dataframe.
Percentage of rows with missing values in segments: 1.19%
--------------------------------------------------------------------------------
As null rows are less than 1%, Dropped rows with missing values in segments dataframe.
New shape of segments after dropping rows with missing values: (4086, 2117)


In [13]:
print("Total number of missing values in segments dataframe after cleaning:")
print(segments.isna().sum().sum())

Total number of missing values in segments dataframe after cleaning:
0


#### Duration of segment correction

In [14]:
# Converting duration from milliseconds to seconds
segments.duration = (segments.duration/1000).round()

In [15]:
# ad duration reference table
ad_duration = pd.DataFrame({
        'brand': ('< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >', '< RemovedDueToNDA >'),
        'duration_s': (30,         30,         15,       30,      30,      30,      15,         15,           30,               15,        15,            15,     15)
        })
ad_duration

,brand,duration_s
0,< RemovedDueToNDA >,30
1,< RemovedDueToNDA >,30
2,< RemovedDueToNDA >,15
3,< RemovedDueToNDA >,30
4,< RemovedDueToNDA >,30
5,< RemovedDueToNDA >,30
6,< RemovedDueToNDA >,15
7,< RemovedDueToNDA >,15
8,< RemovedDueToNDA >,30
9,< RemovedDueToNDA >,15


Clean up ad durations with abnormal values

In [16]:
# count of ad durations in segments
segments[
    (segments['stimuli_type']=='ad') 
    ].value_counts(['duration']).sort_index().reset_index()

,duration,count
0,15.0,265
1,16.0,1236
2,17.0,8
3,19.0,1
4,21.0,1
5,23.0,1
6,28.0,3
7,29.0,1
8,30.0,112
9,31.0,1343


In [17]:
# violation of ad durations cases

TRESHHOLD = 2  # seconds

# checking ad durations
duration_check = segments[(segments['stimuli_type']=='ad') ]\
    .groupby(['duration', 'stimuli_name_clean']).size().reset_index()\
        .merge(
            ad_duration,
            left_on='stimuli_name_clean',
            right_on='brand',
            how='left'
            )
        
duration_check['duration_diff'] = duration_check['duration'] - duration_check['duration_s']


# segments with ad duration violations
violations = segments.merge(
                            duration_check[
                                    abs(duration_check['duration_diff']) > TRESHHOLD
                                    ][['duration', 'stimuli_name_clean']],
                                on=['duration', 'stimuli_name_clean'],
                                how='right'
                            )[['source_file', 'stimuli_name_clean']]\
                                .groupby(['source_file','stimuli_name_clean' ]).size().reset_index()\
                                    .rename(columns={0:'violation_flag'})
                                    
# dropping violating segments
segments = segments.merge(
        violations,
        left_on=['source_file','stimuli_name_clean'],
        right_on=['source_file','stimuli_name_clean'],
        how='left'
)

segments = segments[segments['violation_flag'].isna()].copy()

replace ads durations with the correct ones from ad_duration dataframe

In [18]:
# Merge ad_duration with segments to get the correct durations
segments = segments.merge(
    ad_duration,
    left_on='stimuli_name_clean',
    right_on='brand',
    how='left'
)

# Replace the old duration with the new one
segments['duration'] = segments['duration_s'].fillna(segments['duration'])

# Drop the temporary columns
segments.drop(columns=['brand', 'duration_s', 
                        'violation_flag'], inplace=True)


In [19]:
# Final Check!
segments[
    (segments['stimuli_type']=='ad')
    ].value_counts(['duration']).sort_index().reset_index()

,duration,count
0,15.0,1509
1,30.0,1491


#### Export cleaned segments data

In [20]:
# adding ad_break_session_key to segments
segments['ad_break_session_key'] = segments['key'] + "_" + segments['stimuli_name_clean']

segments_ad_break_session_key = segments['ad_break_session_key'].unique()
print("Number of unique ad_break_session_key in segments:", len(segments_ad_break_session_key))

Number of unique ad_break_session_key in segments: 3544


In [21]:
segments.head()

,source_file,start_ms,end_ms,duration,stimuli_name_clean,stimuli_type,stimuli_type_order,mean-psd_Delta_F7_v^2/Hz,mean-psd_Delta_AF7_v^2/Hz,mean-psd_Delta_Fp1_v^2/Hz,...,z-a-log_EI-2c_right-frontolateral-ext,z-a-log_EI-2d_right-frontolateral-ext,z-a-log_EI-3a_right-frontolateral-ext,z-a-log_EI-3b_right-frontolateral-ext,z-a-log_EI-3c_right-frontolateral-ext,z-a-log_EI-3d_right-frontolateral-ext,z-a-log_AI-1a_right-frontolateral-ext,z-a-log_AI-1b_right-frontolateral-ext,key,ad_break_session_key
0,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,805.8601,389605.8831,389.0,doc,program,program_1,4.087193e-11,2.687389e-11,3.310754e-11,...,-0.131839,-0.644712,-1.003168,-0.758933,-0.728972,-1.410399,-0.177160,-0.233461,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
1,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,389614.8258,420339.2173,30.0,< RemovedDueToNDA >,ad,ad_1,3.839825e-11,2.301769e-11,3.553179e-11,...,1.288322,0.602255,0.987913,0.315851,1.244072,0.379605,-0.309040,-0.284085,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
2,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,420361.0071,435836.4519,15.0,< RemovedDueToNDA >,ad,ad_2,4.567826e-11,2.452332e-11,3.501738e-11,...,-0.627817,0.018828,-0.265536,0.474809,-0.399304,0.381547,-1.184729,-1.231143,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
3,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,435847.3128,467170.9636,30.0,< RemovedDueToNDA >,ad,ad_3,4.156443e-11,5.334442e-11,6.379297e-11,...,1.558769,1.782382,1.691159,1.785671,1.619291,1.519116,-0.743477,-0.619340,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
4,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,467180.9006,497848.9450,30.0,< RemovedDueToNDA >,ad,ad_4,3.390360e-11,2.253874e-11,3.498724e-11,...,-0.870507,-1.329516,-0.976363,-1.181247,-0.878741,-1.094441,1.658208,1.683946,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...


In [22]:
print("Final shape of segments dataframe after cleaning:")
print(segments.shape)

Final shape of segments dataframe after cleaning:
(4070, 2118)


In [23]:
# Exporting cleaned data to CSV files
output_dir = "../../data/processed/eeg/separated/"
os.makedirs(output_dir, exist_ok=True)

segments.to_parquet(os.path.join(output_dir, "segments_obt.parquet"), index=False)

# 4/ Experiment Contents Time Series Data

extracting ads and program time series data from EEG features and merge them in ad_break_level_obt dataframe

In [24]:
ad_break_level_obt = pd.read_parquet("../../data/processed/ad_break_level_meta/ad_break_level_PESR_data.parquet")

print("ad_break_level_obt shape:", ad_break_level_obt.shape)
ad_break_level_obt.head(5)

ad_break_level_obt shape: (3239, 29)


,path,< RemovedDueToNDA >_calculated,< RemovedDueToNDA >,category,subcategory,brand,has_ad,device,ads_break_duration_s,experiment_sequence,...,q_content_engagement,q_content_interest,q_content_quality,q_content_production_quality,q_ad_quality,q_ad_balance,q_ad_creativity,q_ad_engagement,q_ad_brand_influence,unaided_brand_recall
0,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,21,R_8auA8WF4soucDyF,auto,None,< RemovedDueToNDA >,1,laptop,120,2,...,8,8,8,8,5,5,5,4,2,1
1,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,195,R_3y96bLOESrMeMZX,auto,None,< RemovedDueToNDA >,1,smartphone,120,1,...,8,7,3,6,5,3,4,4,4,0
2,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,177,R_2SJF0sXInEcA58d,auto,None,< RemovedDueToNDA >,1,smartphone,120,2,...,4,3,3,5,6,5,4,4,5,1
3,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,165,R_71MkCaPkANLce4b,auto,None,< RemovedDueToNDA >,1,smartphone,120,1,...,9,8,9,9,6,6,6,7,1,1
4,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,129,R_2FjMXZ7PakiuTNn,auto,None,< RemovedDueToNDA >,1,smartphone,120,1,...,10,10,10,10,7,7,6,6,6,1


**double checking the sessions in Segments and Surveys**

In [25]:
# EEG & Surveys keys comparison
temp = pd.DataFrame(eeg['key'].unique(), columns=['eeg_key'])
temp = temp.merge(
    ad_break_level_obt[['path']].drop_duplicates(),
    left_on='eeg_key',
    right_on='path',
    how='outer',
).rename(columns={'path': 'survey_key'})

print("Number of unique EEG keys:", eeg['key'].nunique())
print("Number of unique Segment keys:", segments['key'].nunique())
print("Number of unique Survey keys:", ad_break_level_obt['path'].nunique())

mismatched_keys = temp[temp.isna().any(axis=1)].sort_values(by=['eeg_key', 'survey_key'])
print("Number of mismatched keys between EEG and Surveys:", mismatched_keys.shape[0])

print("-"*90)
print("Number of unique mismatched keys between EEG and Surveys:")
mismatched_keys.nunique()

Number of unique EEG keys: 548
Number of unique Segment keys: 544
Number of unique Survey keys: 583
Number of mismatched keys between EEG and Surveys: 55
------------------------------------------------------------------------------------------
Number of unique mismatched keys between EEG and Surveys:


eeg_key       10
survey_key    45
dtype: int64

In [26]:
# mismatched_keys

In [27]:
ad_break_level_obt['ad_break_session_key'] = ad_break_level_obt['path'] + "_" + ad_break_level_obt['brand']

meta_data_ad_break_session_key = ad_break_level_obt['ad_break_session_key'].unique()
print("Number of unique ad_break_session_key in ad_break_level_obt:", len(meta_data_ad_break_session_key))

Number of unique ad_break_session_key in ad_break_level_obt: 3239


##### Merging time series data with ad_break_level_obt dataframe

In [28]:
experiment_ts = segments.loc[:, ['source_file', 'start_ms', 'end_ms', 'duration', 'stimuli_name_clean', 
                                'stimuli_type', 'stimuli_type_order', 'key', 'ad_break_session_key']
                            ].copy()
# experiment_ts.head()

In [29]:
ad_break_level_obt.shape, experiment_ts.shape

((3239, 30), (4070, 9))

In [30]:
# merging eeg experiment time series with ad_break_level_obt dataframe
new_ad_break_level_obt = ad_break_level_obt.merge(
    experiment_ts,
    left_on=[
        'ad_break_session_key'
    ],
    right_on=[
        'ad_break_session_key'
    ],
    how='inner',   # ========================>>>> Inner because of missmatched sessions! 
)

# dropping unnecessary columns
new_ad_break_level_obt.drop(columns=[
            'key',
            'source_file',
            'stimuli_name_clean',
            'stimuli_type',
            ], 
        inplace=True)

print(f"new_ad_break_level_obt.shape: {new_ad_break_level_obt.shape}")

new_ad_break_level_obt.shape: (2938, 34)


In [31]:
# reordering columns
new_ad_break_level_obt = new_ad_break_level_obt[
        [
        'path',
        '< RemovedDueToNDA >_calculated',
        '< RemovedDueToNDA >',

        'start_ms',
        'end_ms',
        'duration',
        'stimuli_type_order',

        'date',
        'experiment_sequence',

        'category',
        'subcategory',
        'brand',
        'content_watched',

        'ads_break_duration_s',
        'device',

        'n_bev_ads',
        'n_alim_ads',
        'birth_year',

        'age',
        'gender',
        'seen_ad_before',

        'q_content_pleasantness',
        'q_content_engagement',
        'q_content_interest',
        'q_content_quality',
        'q_content_production_quality',
        'q_ad_quality',
        'q_ad_balance',
        'q_ad_creativity',
        'q_ad_engagement',
        'q_ad_brand_influence',

        'has_ad',
        'unaided_brand_recall',
        ]
    ]

new_ad_break_level_obt.head(5)

,path,< RemovedDueToNDA >_calculated,< RemovedDueToNDA >,start_ms,end_ms,duration,stimuli_type_order,date,experiment_sequence,category,...,q_content_interest,q_content_quality,q_content_production_quality,q_ad_quality,q_ad_balance,q_ad_creativity,q_ad_engagement,q_ad_brand_influence,has_ad,unaided_brand_recall
0,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,21,R_8auA8WF4soucDyF,497859.6202,528531.8774,30.0,ad_5,2024-04-18,2,auto,...,8,8,8,5,5,5,4,2,1,1
1,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,195,R_3y96bLOESrMeMZX,475379.8822,505963.3199,30.0,ad_6,2024-05-08,1,auto,...,7,3,6,5,3,4,4,4,1,0
2,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,177,R_2SJF0sXInEcA58d,475016.2027,505649.3255,30.0,ad_6,2024-05-07,2,auto,...,3,3,5,6,5,4,4,5,1,1
3,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,165,R_71MkCaPkANLce4b,497423.3846,528050.6652,30.0,ad_5,2024-05-07,1,auto,...,8,9,9,6,6,6,7,1,1,1
4,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,129,R_2FjMXZ7PakiuTNn,497651.2212,528401.0562,30.0,ad_5,2024-05-03,1,auto,...,10,10,10,7,7,6,6,6,1,1


## Export

In [32]:
# Exporting cleaned data to CSV files
output_dir = "../../data/processed/ad_break_level_meta/"
Path(output_dir).mkdir(parents=True, exist_ok=True)


# Exporting processed_ad_break_level_PESR_data.parquet
new_ad_break_level_obt.to_parquet(os.path.join(output_dir, "temporal_ad_break_level_PESR_data.parquet"), index=False)

In [33]:
new_ad_break_level_obt.shape

(2938, 33)

In [34]:
# Saving the final keys to a CSV file
new_ad_break_level_obt['path'].drop_duplicates().to_csv(
    "../../data/processed/segment_and_survey_final_keys.csv"
)
print("Exported segment_and_survey_final_keys.csv")

final_keys = new_ad_break_level_obt['path'].drop_duplicates().tolist() 
print(f"final count of keys: {len(final_keys)}")
final_keys[:5]

Exported segment_and_survey_final_keys.csv
final count of keys: 534


['E:\\AssoTV\\sensor-data\\O1_raw_20240607\\< RemovedDueToNDA >_lineare_120_Laptop\\001_s21 (1).csv',
 'E:\\AssoTV\\sensor-data\\O1_raw_20240607\\< RemovedDueToNDA >_lineare_120_Smartphone\\001_s195.csv',
 'E:\\AssoTV\\sensor-data\\O1_raw_20240607\\< RemovedDueToNDA >_lineare_120_Smartphone\\005_s177.csv',
 'E:\\AssoTV\\sensor-data\\O1_raw_20240607\\< RemovedDueToNDA >_lineare_120_Smartphone\\009_s165.csv',
 'E:\\AssoTV\\sensor-data\\O1_raw_20240607\\< RemovedDueToNDA >_lineare_120_Smartphone\\015_s129.csv']

In [35]:
# ==== Ad break level ======

#mismarch analysis between segments and ad_break_level_obt dataframes
segments_ad_break_session_key = set(segments['ad_break_session_key'].unique())
meta_data_ad_break_session_key = set(ad_break_level_obt['ad_break_session_key'].unique())

mismatched_ad_break_session_keys = segments_ad_break_session_key.symmetric_difference(meta_data_ad_break_session_key)
print("Number of mismatched ad_break_session_key between segments and ad_break_level_obt:", len(mismatched_ad_break_session_keys))

final_keys_seg_and_meta = segments_ad_break_session_key.intersection(meta_data_ad_break_session_key)
print("Number of final matched ad_break_session_key between segments and ad_break_level_obt:", len(final_keys_seg_and_meta))

Number of mismatched ad_break_session_key between segments and ad_break_level_obt: 907
Number of final matched ad_break_session_key between segments and ad_break_level_obt: 2938


In [36]:
# freeing up memory
del(eeg)
del(experiment_ts)
del(segments)
del(ad_break_level_obt)
del(new_ad_break_level_obt)

# 5/ Bins

In [37]:
print("initial bins shape:", bins.shape)
bins.head()

initial bins shape: (193888, 2120)


,source_file,start_ms,end_ms,bin_param_duration,duration,stimuli_name_clean,stimuli_type,stimuli_type_order,bin_order,ad_order,...,z-a-log_EI-2b_right-frontolateral-ext,z-a-log_EI-2c_right-frontolateral-ext,z-a-log_EI-2d_right-frontolateral-ext,z-a-log_EI-3a_right-frontolateral-ext,z-a-log_EI-3b_right-frontolateral-ext,z-a-log_EI-3c_right-frontolateral-ext,z-a-log_EI-3d_right-frontolateral-ext,z-a-log_AI-1a_right-frontolateral-ext,z-a-log_AI-1b_right-frontolateral-ext,key
1,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,805.8601,3805.8601,3000.0,3000.0,doc,program,program_1,1,<NA>,...,-0.285236,-0.026156,-3.405730,-3.326684,-2.662849,-2.114669,-5.931349,-1.120957,-1.509954,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
2,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,805.8601,5805.8601,5000.0,5000.0,doc,program,program_1,1,<NA>,...,0.107483,-0.146553,-2.872517,-3.191236,-2.251281,-2.214035,-5.442295,-1.349950,-1.740690,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
3,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,3805.8601,6805.8601,3000.0,3000.0,doc,program,program_1,2,<NA>,...,1.894506,0.856918,-0.073149,-1.246899,-0.087731,-0.805335,-2.529545,-2.063084,-2.231604,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
4,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,5805.8601,10805.8601,5000.0,5000.0,doc,program,program_1,2,<NA>,...,3.119100,1.799718,-0.277707,-0.237048,2.040002,0.988895,-1.510551,-1.908700,-1.967045,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
5,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,6805.8601,9805.8601,3000.0,3000.0,doc,program,program_1,3,<NA>,...,3.637167,0.681683,-2.172027,-1.210887,3.169088,0.377664,-2.320337,-1.612684,-1.842413,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...


Filtering the source files we need for bins level analysis (out of final keys)

In [38]:
available_bins = bins[bins['key'].isin(final_keys)].copy()

#temporary de< RemovedDueToNDA > bins dataframe to save memory
del(bins)

In [39]:
# split bins of 3 and 5 seconds

# bins_3s = available_bins[available_bins['bin_param_duration'] == 3000].copy()
# print(f"shape of 3s bins: {bins_3s.shape}")

bins_5s = available_bins[available_bins['bin_param_duration'] == 5000].copy()
print(f"shape of 5s bins: {bins_5s.shape}")

del(available_bins)

shape of 5s bins: (71766, 2120)


de< RemovedDueToNDA > unneeded rows (end part of shows)

In [40]:
# we don't need the program 2 (the rest of show is not important)
bins_5s = bins_5s[bins_5s['stimuli_type_order'] != 'program_2'].copy()

In [41]:
bins_5s.head()

,source_file,start_ms,end_ms,bin_param_duration,duration,stimuli_name_clean,stimuli_type,stimuli_type_order,bin_order,ad_order,...,z-a-log_EI-2b_right-frontolateral-ext,z-a-log_EI-2c_right-frontolateral-ext,z-a-log_EI-2d_right-frontolateral-ext,z-a-log_EI-3a_right-frontolateral-ext,z-a-log_EI-3b_right-frontolateral-ext,z-a-log_EI-3c_right-frontolateral-ext,z-a-log_EI-3d_right-frontolateral-ext,z-a-log_AI-1a_right-frontolateral-ext,z-a-log_AI-1b_right-frontolateral-ext,key
2,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,805.8601,5805.8601,5000.0,5000.0,doc,program,program_1,1,<NA>,...,0.107483,-0.146553,-2.872517,-3.191236,-2.251281,-2.214035,-5.442295,-1.349950,-1.740690,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
4,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,5805.8601,10805.8601,5000.0,5000.0,doc,program,program_1,2,<NA>,...,3.119100,1.799718,-0.277707,-0.237048,2.040002,0.988895,-1.510551,-1.908700,-1.967045,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
7,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,10805.8601,15805.8601,5000.0,5000.0,doc,program,program_1,3,<NA>,...,1.757433,3.565282,1.277831,1.359586,0.732951,2.834359,-0.166243,0.572540,0.776246,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
10,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,15805.8601,20805.8601,5000.0,5000.0,doc,program,program_1,4,<NA>,...,2.779312,1.841002,2.560159,1.596225,2.158188,1.424904,1.469279,-3.364040,-3.189203,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
12,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,20805.8601,25805.8601,5000.0,5000.0,doc,program,program_1,5,<NA>,...,1.038946,1.840102,0.134628,1.483745,1.163691,2.037841,0.307296,0.074604,0.097508,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...


#### inital inspection

In [42]:
bins_5s.info()
# dtypes: Int64(12), float64(2100), object(3)

# bins_5s.select_dtypes(include=['object']).columns

<class 'pandas.core.frame.DataFrame'>
Index: 57941 entries, 2 to 197953
Columns: 2120 entries, source_file to key
dtypes: Int64(14), float64(2101), object(5)
memory usage: 938.4+ MB


In [43]:
print("Number of duplicated rows in bin dataframe:")
print(bins_5s.duplicated().sum())

Number of duplicated rows in bin dataframe:
0


### null values Analysis

In [44]:
print("columns wise missing values in segments dataframe:")
print(f"{(bins_5s.isna().sum() > 0).sum()} columns with missing values in segments dataframe.")
print()

print("Top 10 columns with highest percentage of missing values:")
print(round((bins_5s.isna().sum() / len(bins_5s) * 100).sort_values(ascending=False).head(10), 1))
print('-'*80)

print("row wise missing values in segments dataframe:")
print(f"Percentage of rows with missing values in segments: \
        {(bins_5s.drop(columns=['ad_order']).isna().any(axis=1).sum() / len(bins_5s) * 100):.2f} %")
print('-'*80)

columns wise missing values in segments dataframe:
2110 columns with missing values in segments dataframe.

Top 10 columns with highest percentage of missing values:
ad_order                       71.8
z-p_GFP_Theta_All               4.7
z-a_GFP_Theta-lower_All         4.7
z-p-log_GFP_Gamma_All           4.7
GFP_Delta_All                   4.7
z-a_GFP_Delta_All               4.7
z-a_GFP_Theta_All               4.7
z-a-log_GFP_Gamma_All           4.7
z-a-log_GFP_Beta_All            4.7
z-a-log_GFP_Alpha-upper_All     4.7
dtype: float64
--------------------------------------------------------------------------------
row wise missing values in segments dataframe:
Percentage of rows with missing values in segments:         4.73 %
--------------------------------------------------------------------------------


> We've 5% of null values, but we can't just drop them, as they will ruin the time series related to a session. We need to a deeper analysis of the sessions with missing values.

In [45]:
nv_report = \
    bins_5s.groupby(['key', 'stimuli_type_order']).apply(
            lambda x: pd.Series({
                'null_rows_count': x.drop(columns=['ad_order']).isna().any(axis=1).sum(),
                'total_rows': len(x),
                'null_rows_percentage': round(x.drop(columns=['ad_order']).isna().any(axis=1).sum() / len(x) * 100, 1)
            }),
            
            include_groups=False
        ).reset_index()

nv_report = nv_report[nv_report['null_rows_count'] > 0].copy()


total_rows_ad , total_rows_program = bins_5s[bins_5s['stimuli_type'] == 'ad'].shape[0] , bins_5s[bins_5s['stimuli_type'] == 'program'].shape[0]
total_null_cases_ad, total_null_cases_program  = bins_5s[(bins_5s['key'].isin(nv_report.key.unique())) & (bins_5s['stimuli_type_order'].str.startswith('ad'))].shape[0], \
    bins_5s[(bins_5s['key'].isin(nv_report.key.unique())) & (bins_5s['stimuli_type_order'].str.startswith('program'))].shape[0] 



print("Segments with null cases report:")
print("Ad bin:", round(total_null_cases_ad / total_rows_ad * 100), "%")
print("Program bin:", round(total_null_cases_program / total_rows_program * 100), "%")
print('-'*20)
print("total segments:", bins_5s[['key', 'stimuli_type_order']].drop_duplicates().shape[0])
print("segments with null cases:", bins_5s[(bins_5s['key'].isin(nv_report.key.unique()))][['key', 'stimuli_type_order']].drop_duplicates().shape[0])

Segments with null cases report:
Ad bin: 90 %
Program bin: 87 %
--------------------
total segments: 3500
segments with null cases: 3145


#### Null Values Analysis per session/segment

1/ ads

    - impute if possible 
    - drop the entire ad segment if not possible
    - drop the entire session if too many ad segments are missing => Not Going for that! as our data input is ad level! 


2/ programs

    only the last part is important! like last 10 seconds before the program ends


<!-- **Strategies for Handling Missing Values in Bins**

There are three approaches to dealing with missing values in the bins dataset:

1. **Drop the entire session** - Remove all segments associated with a session containing missing values
2. **Drop only the affected segment** - Remove only the specific segment with missing values while retaining other segments from the same session
3. **Impute missing values** - Fill in missing values using interpolation or other statistical methods

**Considerations:**
- Dropping entire sessions ensures data integrity but may result in significant data loss
- Dropping only segments preserves more data but may create gaps in the time series
- Imputation maintains continuity but introduces estimated values that may affect analysis accuracy

The choice depends on:
- The percentage of missing values per session/segment
- The distribution pattern of missing values (random vs. consecutive)
- The downstream analysis requirements -->

##### Segment-wise Missing Values Analysis

In [46]:
segment_key = ['key', 'stimuli_type_order', 'stimuli_type']

print("Number of segments in 5s bins dataframe:", bins_5s.groupby(segment_key).size().shape)

Number of segments in 5s bins dataframe: (3500,)


In [47]:
# Calculating null values report per segment based on the segment Key

nv_report_segment = \
    bins_5s.groupby(segment_key).apply(
            lambda x: pd.Series({
                'null_rows_count': x.drop(columns=['ad_order']).isna().any(axis=1).sum(),
                'total_rows': len(x),
                'null_rows_percentage': round(x.drop(columns=['ad_order']).isna().any(axis=1).sum() / len(x) * 100, 1)
            }),
            
            include_groups=False
        ).reset_index()

nv_report_segment['category'] = pd.cut(
    nv_report_segment['null_rows_percentage'],
    bins=[-1, 5, 10, 30, 50, 100],
    labels=['0-5%', '5-10%', '10-30%', '30-50%', '50-100%']
)

nv_report_segment = nv_report_segment[nv_report_segment['null_rows_count'] > 0].copy()

print('Preview of nv_report_segment:')
print(nv_report_segment.shape)
display(nv_report_segment.head())
print('-'*80)
print()

# REPORT
print("Null Values Report per Segment:")
print("-- proportion of segments, grouped by stimuli type --")
display(
    pd.crosstab(
        nv_report_segment['category'],
        nv_report_segment['stimuli_type'],
        normalize='columns',
        margins=True
    ).round(2).multiply(100)\
        .reset_index()\
        .rename(columns={'category': 'Null Values Percentage'})
    )
print('* each cell represents the percentage of segments with corresponding percentage of null values in bins, grouped by stimuli type.')


Preview of nv_report_segment:
(1464, 7)


,key,stimuli_type_order,stimuli_type,null_rows_count,total_rows,null_rows_percentage,category
1,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,ad_2,ad,1.0,4.0,25.0,10-30%
2,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,ad_3,ad,1.0,7.0,14.3,10-30%
4,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,ad_5,ad,1.0,7.0,14.3,10-30%
5,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,program_1,program,1.0,78.0,1.3,0-5%
6,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,ad_1,ad,1.0,7.0,14.3,10-30%


--------------------------------------------------------------------------------

Null Values Report per Segment:
-- proportion of segments, grouped by stimuli type --


stimuli_type,Null Values Percentage,ad,program,All
0,0-5%,0.0,70.0,16.0
1,5-10%,0.0,15.0,4.0
2,10-30%,89.0,14.0,72.0
3,30-50%,7.0,1.0,5.0
4,50-100%,4.0,0.0,3.0


* each cell represents the percentage of segments with corresponding percentage of null values in bins, grouped by stimuli type.


Drilling down to find a treshhold for dropping segments based on missing values percentage

In [48]:
# Ads segments null values distribution
print("\n-- Ads segments null values distribution \n based on total rows and null rows count --")
nv_report_segment[nv_report_segment['stimuli_type']== 'ad'] \
    .value_counts(['total_rows','null_rows_count']).reset_index().sort_values(by=['null_rows_count', 'total_rows',])


-- Ads segments null values distribution 
 based on total rows and null rows count --


,total_rows,null_rows_count,count
0,4.0,1.0,503
11,6.0,1.0,1
1,7.0,1.0,449
14,10.0,1.0,1
16,11.0,1.0,1
17,12.0,1.0,1
3,4.0,2.0,56
12,5.0,2.0,1
2,7.0,2.0,58
13,8.0,2.0,1


> Treshhold of dropping missing values: **30%** of the segment's rows being null

same analysis for program segments has been done. the min and max total rows and the null counts are not comparable, so 30 percent is still fare enough

##### Another look at segments with null cases

###### Null Positions Analysis

index of missing rows per segment 

In [49]:
# Create bin position category based on bin_order per segment
bins_5s['bin_position'] = bins_5s.groupby(segment_key)['bin_order'].transform(
    lambda x: pd.Series(['first'] + ['middle'] * (len(x) - 2) + ['last'] if len(x) > 2 
                       else (['first', 'last'] if len(x) == 2 
                             else ['first']), index=x.index)
)

# Calculating null values report per index of missing rows per segment 
nv_report_index = \
    bins_5s.groupby(segment_key + ['bin_order', 'bin_position'], group_keys=True).apply(
            lambda x: pd.Series({
                'is_null': x.drop(columns=['ad_order']).isna().any(axis=1).sum()
            }),
            
            include_groups=False
        ).reset_index()


nv_report_index = nv_report_index[nv_report_index['is_null'] > 0].copy()

print('Preview of nv_report_segment:')
display(nv_report_index.head())
print('-'*80)
print()


# REPORT
print("Index of Null Values Report per Segment:")
print("-- reports the position of null values within each segment and stimuli type --")

display(
    pd.crosstab(
        nv_report_index['bin_position'],
        nv_report_index['stimuli_type'],
        normalize='columns',
        margins=True
    ).round(2).multiply(100)\
        .reset_index()
)



Preview of nv_report_segment:


,key,stimuli_type_order,stimuli_type,bin_order,bin_position,is_null
10,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,ad_2,ad,4,last,1
17,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,ad_3,ad,7,last,1
31,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,ad_5,ad,7,last,1
109,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,program_1,program,78,last,1
116,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,ad_1,ad,7,last,1


--------------------------------------------------------------------------------

Index of Null Values Report per Segment:
-- reports the position of null values within each segment and stimuli type --


stimuli_type,bin_position,ad,program,All
0,first,5.0,2.0,4.0
1,last,74.0,15.0,46.0
2,middle,20.0,82.0,50.0


###### Consecutive Nulls Analysis

In [50]:
bins_5s.sort_values(by= ['key', 'start_ms'], inplace=True)
# missing indicator
bins_5s['has_missing'] = bins_5s.drop(columns=['ad_order']).isna().any(axis=1).astype(int)

# id for groups of continuous missing values
grp = (bins_5s['has_missing'] == 0).cumsum()


# consecutive cumulative missing counter
bins_5s['consecutive_missing_index'] = (
    bins_5s['has_missing']
    .groupby([grp,bins_5s['stimuli_type_order']])
    .cumsum()
)

# len of consecutive missing counter per group
bins_5s['consecutive_missing_count'] = (
    bins_5s['consecutive_missing_index']
    .groupby([grp, bins_5s['stimuli_type_order']])
    .transform('max')
) * bins_5s['has_missing']


# Calculating null values report per index of missing rows per segment
nv_report_consecutive_missing = (bins_5s[
                                    # (bins_5s['bin_position'] == 'middle') &
                                    (bins_5s['consecutive_missing_count'] > 0)
                                    ].groupby([grp, 'consecutive_missing_count', 'stimuli_type'])['has_missing'].mean()).reset_index(name='count').rename(columns={'has_missing': 'grp'})


print('Preview of nv_report_consecutive_missing:')
display(nv_report_consecutive_missing.head())
print('-'*80)
print()

#report
# print('-for middle positions only-\n')
print('Consective nulls in segments')
# nv_report_consecutive_missing.value_counts('consecutive_missing_count', normalize=True).sort_index().reset_index().rename(columns={ 0: 'consecutive_missing_count', 'consecutive_missing_count': 'proportion'}).round(2)
display(
    pd.crosstab(
        nv_report_consecutive_missing['consecutive_missing_count'],
        nv_report_consecutive_missing['stimuli_type'],
        normalize='columns',
        margins=True
    ).round(3).multiply(100)\
        .reset_index()
)


Preview of nv_report_consecutive_missing:


,grp,consecutive_missing_count,stimuli_type,count
0,77,1,program,1.0
1,87,1,ad,1.0
2,93,1,ad,1.0
3,106,1,ad,1.0
4,181,1,program,1.0


--------------------------------------------------------------------------------

Consective nulls in segments


stimuli_type,consecutive_missing_count,ad,program,All
0,1,91.8,83.2,88.0
1,2,5.3,11.2,7.9
2,3,1.3,2.6,1.9
3,4,1.2,1.0,1.1
4,5,0.1,0.5,0.3
5,6,0.1,0.2,0.1
6,7,0.2,0.3,0.3
7,8,0.0,0.3,0.1
8,9,0.0,0.2,0.1
9,10,0.0,0.1,0.0


Extrapolating would save 95% of null cases. 

--- VISAUAL INSPECTION OF MISSING VALUES IN BINS ---

In [51]:
alpha_cols = [
    'mean-psd_Alpha_F7_v^2/Hz',
    'mean-psd_Alpha_AF7_v^2/Hz',
    'mean-psd_Alpha_Fp1_v^2/Hz',
    'mean-psd_Alpha_Fpz_v^2/Hz',
    'mean-psd_Alpha_Fp2_v^2/Hz',
    'mean-psd_Alpha_AF8_v^2/Hz',
    'mean-psd_Alpha_F8_v^2/Hz',
]
psd_cols = [col for col in bins_5s.columns if col.startswith('mean-psd_') and '_All_v^2/Hz' in col and not 'Delta' in col]

import random
user_keys = random.sample(final_keys, 2)

for user_key in user_keys:

    print(f"Plotting PSD Alpha for user key: {user_key}")

    df = bins_5s[bins_5s['key'] == user_key].loc[:, ['key', 'start_ms'] + psd_cols]

    df_melt = df.melt(
        id_vars=['key', 'start_ms'],
        value_vars=psd_cols,
        var_name='EEG Channel',
        value_name='PSD (v^2/Hz)'
    )

    # 🔥 IMPORTANT: sort by time to allow gaps
    df_melt = df_melt.sort_values("start_ms")

    plt.figure(figsize=(20, 4))

    sns.lineplot(
        data=df_melt,
        x="start_ms",
        y="PSD (v^2/Hz)",
        hue="EEG Channel",
        alpha=0.6
    )

    # 🔴 mark missing values
    missing = df_melt[df_melt["PSD (v^2/Hz)"].isna()]
    plt.scatter(
        missing["start_ms"],
        [0] * len(missing),
        color="red",
        s=25,
        label="Missing"
    )

    plt.title(f'PSD Over Time — {user_key}')
    plt.xlabel('Start Time (ms)')
    plt.ylabel('PSD (v^2/Hz)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

Plotting PSD Alpha for user key: E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_lineare_180_Smartphone_DA USARE\011_s267.csv


Plotting PSD Alpha for user key: E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_lineare_120_Smartphone\010_s162.csv


#### Imputation of missing values startegy:

1. De< RemovedDueToNDA > ad segments with more than 30% missing values
2. For the remaining missing values in ad segments, impute using the interpolation method with a limit 

    We fill missing values using linear interpolation between the nearest valid values before and after the gap.
    We apply this separately for each segment, sorted by time.

    We also introduce a gap limit (max_gap) to make sure we only interpolate short missing stretches, leaving long gaps untouched.



In [52]:
bins_5s_cleaned = bins_5s.sort_values(['key', 'start_ms']).copy()
print("Shape of bins_5s before cleaning:", bins_5s_cleaned.shape)

# Step 1 - Deleting ad segments with more than 30% missing values
print("="*80)
print("step 1 - Removing ad segments with more than 30% missing values")
TRESHHOLD = 0.3  # 30%

flagged_ad_segments = nv_report_segment[
                (nv_report_segment['stimuli_type'] == 'ad') &
                (nv_report_segment['null_rows_percentage'] > TRESHHOLD * 100)].loc[:, ['key', 'stimuli_type_order']]

bins_5s_cleaned = bins_5s[~bins_5s.set_index(['key', 'stimuli_type_order']).index.isin(flagged_ad_segments.set_index(['key', 'stimuli_type_order']).index)].sort_values(['key', 'start_ms'])
print("Shape of bins_5s after removing flagged ad segments:", bins_5s_cleaned.shape)



# Step 2 - Imputation of missing values in bins_5s dataframe for ADS with treshhold of 3
print("="*80)
print("step 2 - Imputation of missing values in bins_5s dataframe")
max_gap = 15  # maximum allowed consecutive missing values

value_cols = [col for col in bins_5s.columns if col not in ['source_file', 'start_ms', 'end_ms', 'bin_param_duration', 'duration', 
                                                            'stimuli_name_clean', 'stimuli_type', 'stimuli_type_order', 'bin_order',
                                                            'ad_order', 'key', 'has_missing', 'consecutive_missing_index', 'bin_position']]

bins_5s_cleaned[value_cols] = (
    bins_5s_cleaned.groupby('key')[value_cols]
        .apply(lambda g: g.interpolate(
            method='linear',
            limit=max_gap,         
            limit_direction='both'
        ))
        .reset_index(level=0, drop=True)
)
print("Shape of bins_5s after imputation:", bins_5s_cleaned.shape)



Shape of bins_5s before cleaning: (57941, 2124)
step 1 - Removing ad segments with more than 30% missing values
Shape of bins_5s after removing flagged ad segments: (57350, 2124)
step 2 - Imputation of missing values in bins_5s dataframe
Shape of bins_5s after imputation: (57350, 2124)


In [53]:
# checking null status after imputation
print("columns wise missing values in segments dataframe:")
print(f"{(bins_5s_cleaned.isna().sum() > 0).sum()} columns with missing values in segments dataframe.")
print()

print("Top 10 columns with highest percentage of missing values:")
print(round((bins_5s_cleaned.isna().sum() / len(bins_5s_cleaned) * 100).sort_values(ascending=False).head(10), 1))
print('-'*80)

print("row wise missing values in segments dataframe:")
print(f"Percentage of rows with missing values in segments: \
        {(bins_5s_cleaned.drop(columns=['ad_order']).isna().any(axis=1).sum() / len(bins_5s_cleaned) * 100):.2f} %")

print("\n", "="*80)
print('Segments null values report after cleaning:')

# 1) Precompute: detect rows that have ANY NaN (except ad_order)
mask_null = bins_5s_cleaned.drop(columns=['ad_order']).isna().any(axis=1)

# 2) Aggregate per segment (FAST)
nv_report_cleaned = (
    bins_5s_cleaned
    .assign(has_null=mask_null)
    .groupby(['key', 'stimuli_type_order'])
    .agg(
        null_rows_count=('has_null', 'sum'),
        total_rows=('has_null', 'size'),
    )
    .assign(
        null_rows_percentage=lambda x: (x.null_rows_count / x.total_rows * 100).round(1)
    )
    .reset_index()
)

nv_report_cleaned = nv_report_cleaned[nv_report_cleaned['null_rows_count'] > 0].copy()

# 3) Compute % of bins with nulls by type
total_rows_ad  = bins_5s_cleaned.query("stimuli_type == 'ad'").shape[0]
total_rows_program = bins_5s_cleaned.query("stimuli_type == 'program'").shape[0]

null_keys = nv_report_cleaned['key'].unique()

total_null_cases_ad = bins_5s_cleaned[
    (bins_5s_cleaned['key'].isin(null_keys)) &
    (bins_5s_cleaned['stimuli_type_order'].str.startswith('ad'))
].shape[0]

total_null_cases_program = bins_5s_cleaned[
    (bins_5s_cleaned['key'].isin(null_keys)) &
    (bins_5s_cleaned['stimuli_type_order'].str.startswith('program'))
].shape[0]

# Report
print("Segments with null cases report:")
print("Ad bin:", round(total_null_cases_ad / total_rows_ad * 100), "%")
print("Program bin:", round(total_null_cases_program / total_rows_program * 100), "%")

print('-' * 20)
total_segments = bins_5s_cleaned[['key', 'stimuli_type_order']].drop_duplicates().shape[0]

segments_with_null = (
    bins_5s_cleaned[bins_5s_cleaned['key'].isin(null_keys)]
    [['key', 'stimuli_type_order']]
    .drop_duplicates()
    .shape[0]
)

print("total segments:", total_segments)
print("segments with null cases:", segments_with_null)

columns wise missing values in segments dataframe:
1 columns with missing values in segments dataframe.

Top 10 columns with highest percentage of missing values:
ad_order                                                 72.5
source_file                                               0.0
z-p-log_mean-psd_Gamma_frontal-midline_v^2/Hz             0.0
z-p-log_mean-psd_Gamma_prefrontal_v^2/Hz                  0.0
z-p-log_mean-psd_Gamma_anterior-frontal_v^2/Hz            0.0
z-p-log_mean-psd_Gamma_frontolateral_v^2/Hz               0.0
z-p-log_mean-psd_Gamma_frontolateral-ext_v^2/Hz           0.0
z-p-log_mean-psd_Gamma_left-frontolateral-ext_v^2/Hz      0.0
z-p-log_mean-psd_Gamma_right-frontolateral-ext_v^2/Hz     0.0
z-p-log_mean-psd_Beta_right-frontolateral-ext_v^2/Hz      0.0
dtype: float64
--------------------------------------------------------------------------------
row wise missing values in segments dataframe:
Percentage of rows with missing values in segments:         0.00 %

Segmen

### Duration Fixing

In [ ]:
bins_5s.head()

In [55]:
#sanity chekc of duration
print("number of mismatched duration rows in bins_5s_cleaned:",
(((bins_5s['end_ms'] - bins_5s['start_ms']) / 1000).round() != ((bins_5s['duration'] / 1000).round() )).sum()
)

number of mismatched duration rows in bins_5s_cleaned: 0


In [ ]:
durations = bins_5s_cleaned.groupby(['key', 'stimuli_name_clean', 'stimuli_type']).agg(
                                    duration = ('duration', 'sum')
                                ).reset_index()
durations['duration'] = (durations['duration'] / 1000).round(1)

display(durations.head())


# Boxplot of durations per stimuli_name_clean for ads and programs
plt.figure(figsize=(20, 3))
sns.boxplot(
    data=durations[durations['stimuli_type'] == 'ad'],
    y='duration',
    x='stimuli_name_clean'
)
plt.xticks(rotation=90, fontsize=8)
plt.show()


plt.figure(figsize=(20, 3))
sns.boxplot(
    data=durations[durations['stimuli_type'] == 'program'],
    y='duration',
    x='stimuli_name_clean'
)
plt.xticks(rotation=90, fontsize=8)
plt.show()

< RemovedDueToNDA >

In [57]:
# highlights the rows where mean and the value differ significantly (>2s) in duration
durations['mean_stimuli'] = durations.groupby('stimuli_name_clean')['duration'].transform('mean').round(2)
durations['diff_mean_of_sti_'] = abs(durations['duration'] - durations['mean_stimuli']).round(2)
durations[durations['diff_mean_of_sti_'] > 2.0]


,key,stimuli_name_clean,stimuli_type,duration,mean_stimuli,diff_mean_of_sti_
58,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >,ad,47.1,15.81,31.29
555,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >,ad,27.8,31.31,3.51
565,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,le iene inside,program,385.1,365.92,19.18
677,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >,ad,27.9,15.66,12.24
682,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >,ad,28.6,15.69,12.91
743,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,le iene inside,program,385.0,365.92,19.08
762,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,le iene inside,program,385.0,365.92,19.08
765,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >,ad,95.6,16.11,79.49
793,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,le iene inside,program,385.1,365.92,19.18
801,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >,ad,91.6,16.00,75.60


### Export cleaned bin data

In [ ]:
bins_5s_cleaned.head()

In [59]:
# Final cleanup - dropping helper columns
bins_5s_cleaned.drop(
    columns=[
        'bin_position',
        'consecutive_missing_count',
        'consecutive_missing_index',
        'has_missing',
        ],
    inplace=True
    )

In [60]:
print("Final shape of bin dataframe after cleaning:")
print(bins_5s_cleaned.shape)

Final shape of bin dataframe after cleaning:
(57350, 2120)


In [61]:
# Exporting cleaned data to CSV files
output_dir = "../../data/processed/eeg/separated/"
os.makedirs(output_dir, exist_ok=True)

bins_5s_cleaned.to_parquet(os.path.join(output_dir, "bin5s_obt.parquet"), index=False)

In [62]:
# Saving the final keys to a CSV file
bins_5s_cleaned[['key', 'stimuli_name_clean']].drop_duplicates().to_csv(
    "../../data/processed/final_segments_keys.csv",
    index=False
)
print("Exported final_segments_keys.csv")

bins_5s_cleaned[['key', 'stimuli_name_clean']].drop_duplicates().head()

Exported final_segments_keys.csv


,key,stimuli_name_clean
2,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,doc
211,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >
230,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >
241,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >
260,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,< RemovedDueToNDA >


# Ad break session key analysis and merging

In [63]:
# ==== Ad break level ======

mismatched_ad_break_session_keys = segments_ad_break_session_key.symmetric_difference(meta_data_ad_break_session_key)
print("Number of mismatched ad_break_session_key between segments and ad_break_level_obt:", len(mismatched_ad_break_session_keys))

final_keys_seg_and_meta = segments_ad_break_session_key.intersection(meta_data_ad_break_session_key)
print("Number of final matched ad_break_session_key between segments and ad_break_level_obt:", len(final_keys_seg_and_meta))

Number of mismatched ad_break_session_key between segments and ad_break_level_obt: 907
Number of final matched ad_break_session_key between segments and ad_break_level_obt: 2938


In [64]:
bins_5s_cleaned['ad_break_session_key'] = bins_5s_cleaned['key'] + "_" + bins_5s_cleaned['stimuli_name_clean']

bins_5s_ad_break_session_key = set(bins_5s_cleaned['ad_break_session_key'].unique())
print("Number of unique ad_break_session_key in cleaned bins_5s:", len(bins_5s_ad_break_session_key))

mismatched_ad_break_session_keys_bins = bins_5s_ad_break_session_key.symmetric_difference(final_keys_seg_and_meta)
print("Number of mismatched ad_break_session_key between cleaned bins_5s and final matched segments & meta:", len(mismatched_ad_break_session_keys_bins))

final_keys_seg_and_meta_and_bins = bins_5s_ad_break_session_key.intersection(final_keys_seg_and_meta)
print("Number of final matched ad_break_session_key between cleaned bins_5s and final matched segments & meta:", len(final_keys_seg_and_meta_and_bins))

Number of unique ad_break_session_key in cleaned bins_5s: 3383
Number of mismatched ad_break_session_key between cleaned bins_5s and final matched segments & meta: 651
Number of final matched ad_break_session_key between cleaned bins_5s and final matched segments & meta: 2835


/var/folders/kh/21c5gvns57q5bq666b4_v4hh0000gn/T/ipykernel_93565/569402374.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  bins_5s_cleaned['ad_break_session_key'] = bins_5s_cleaned['key'] + "_" + bins_5s_cleaned['stimuli_name_clean']


In [65]:
#saving final ad break session keys 
list(final_keys_seg_and_meta_and_bins)

pd.DataFrame(list(final_keys_seg_and_meta_and_bins), columns=['ad_break_session_key'] ).to_csv(
    "../../data/final_ad_break_session_keys.csv",
    index=False
)
print("Exported final_ad_break_session_keys.csv")

Exported final_ad_break_session_keys.csv
